# Description

This notebook implements the first of the two main stages of the Retrieval-Augmented Generation (RAG) pipeline: data indexing.

The dataset consists of FIA Formula 1 Sporting Regulations documents covering the 2018–2026 seasons. These documents are processed and transformed into vector representations to enable efficient semantic retrieval.

The data indexing pipeline includes the following steps:

1. Document Chunking.
2. Creating vector database.
3. Creating embeddings
4. Saving embeddings into database

Data source: https://api.fia.com/regulation/category/110

## Import libraries

In [1]:
import json
import fitz
import re
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, SparseVectorParams, PointStruct
from fastembed import SparseTextEmbedding, TextEmbedding
from uuid import uuid4

## 1. Chunking

In [2]:
# load metadata about PDFs

with open("../data/documents/metadata.json", "r") as f:
    metadata = json.load(f)

In [3]:
print(json.dumps(metadata, indent=4, ensure_ascii=False))

{
    "1-2018_sporting_regulations_2018-07-17.pdf": {
        "year": 2018,
        "first_page": 2,
        "last_page": 40
    },
    "2019_sporting_regulations_-_2019-03-12.pdf": {
        "year": 2019,
        "first_page": 2,
        "last_page": 41
    },
    "2020_formula_1_sporting_regulations_-_iss_14_-_2020-11-23.pdf": {
        "year": 2020,
        "first_page": 2,
        "last_page": 44
    },
    "2021_formula_1_sporting_regulations_-_iss_13_-_2021-12-08.pdf": {
        "year": 2021,
        "first_page": 2,
        "last_page": 56
    },
    "fia_2022_formula_1_sporting_regulations_-_issue_9_-_2022-10-19_0.pdf": {
        "year": 2022,
        "first_page": 2,
        "last_page": 68
    },
    "fia_2023_formula_1_sporting_regulations_-_issue_8_-_2023-12-06_0.pdf": {
        "year": 2023,
        "first_page": 2,
        "last_page": 69
    },
    "fia_2024_formula_1_sporting_regulations_-_issue_7_-_2024-07-31.pdf": {
        "year": 2024,
        "first_page": 2,
     

Since the PDF documents contain introduction pages and final pages that do not include regulation content, metadata was created for each PDF file. The metadata includes the first page where the regulations begin, the last page where they end, and the year of the corresponding document.

In [4]:
# function to extract text from PDFs

def extract_text(document_path, first_page, last_page):

    doc = fitz.open(document_path)


    pages = []

    for page_num, page in enumerate(doc):
        text = page.get_text("text")

        pages.append({
            "page": page_num + 1,
            "text": text
        })


    full_text = ''

    for page in pages:

        if page['page'] < first_page:
            continue

        if page['page'] > last_page:
            break

        full_text += page['text'] + '\n'


    return full_text

In [5]:
# Function to parse 2018–2025 PDFs into chunks with metadata

def parse_f1_regulations_2018_2025(full_text): 


    # =====================================================
    # TEXT CLEANING
    # =====================================================

    # Normalize line endings
    full_text = full_text.replace("\r", "\n")

    # Remove document titles
    full_text = re.sub(
        r'\b20\d{2}\s+(?:F1|Formula\s+1)\s+Sporting\s+Regulations\b',
        '',
        full_text,
        flags=re.IGNORECASE
    )

    # Remove page counters
    full_text = re.sub(
        r'\b\d+/\d+\b',
        '',
        full_text
    )

    # Remove dates
    full_text = re.sub(
        r'\b\d{1,2}\s+[A-Za-z]+\s+\d{4}\b',
        '',
        full_text
    )

    # Remove FIA copyright footer
    full_text = re.sub(
        r'©\d{4}[^\n]*Automobile',
        '',
        full_text,
        flags=re.IGNORECASE
    )

    # Remove issue numbers
    full_text = re.sub(
        r'Issue\s+\d+',
        '',
        full_text,
        flags=re.IGNORECASE
    )

    # Fix OCR cases where a new article was merged
    full_text = re.sub(
        r'([a-zA-Z\.\)])\s+(\d+\.\d+)\s+([A-Z])',
        r'\1\n\2\n\3',
        full_text
    )

    # Fix OCR cases where a new chapter was merged
    full_text = re.sub(
        r'([a-zA-Z\.\)])\s+(\d+\)\s+[A-Z])',
        r'\1\n\2',
        full_text
    )



    # =====================================================
    # CHAPTER DETECTION
    # =====================================================

    chapters = []

    for match in re.finditer(
        r'(\d+)\)\s*\n+\s*([A-Z][A-Z0-9\s,&\-\(\)/]+)',
        full_text
    ):

        chapter_name = re.sub(
            r'\s+',
            ' ',
            match.group(2)
        ).strip()

        # Remove chapter number accidentally attached
        chapter_name = re.sub(
            r'\s+\d+$',
            '',
            chapter_name
        ).strip()

        chapters.append({
            "start": match.start(),
            "chapter": chapter_name
        })



    # =====================================================
    # ARTICLE DETECTION
    # =====================================================

    article_matches = list(
        re.finditer(
            r'(?m)^\s*(\d+\.\d+)\s*$',
            full_text
        )
    )

    chunks = []

    for idx, article_match in enumerate(article_matches):

        article = article_match.group(1)

        # Start immediately after article number
        start = article_match.end()

        # Next article position
        next_article = (
            article_matches[idx + 1].start()
            if idx < len(article_matches) - 1
            else len(full_text)
        )

        # Next chapter position
        next_chapter = len(full_text)

        for chapter in chapters:
            if chapter["start"] > article_match.start():
                next_chapter = chapter["start"]
                break

        end = min(next_article, next_chapter)

        # Extract article content
        content = full_text[start:end]

        # Find active chapter
        current_chapter = None

        for chapter in chapters:
            if chapter["start"] < article_match.start():
                current_chapter = chapter["chapter"]
            else:
                break



        # =================================================
        # CONTENT CLEANUP
        # =================================================

        content = re.sub(r'\s+', ' ', content)

        # Remove chapter title accidentally attached
        content = re.sub(
            r'\s+\d+\)\s+[A-Z][A-Z0-9\s,&\-\(\)/]+$',
            '',
            content
        )

        content = content.strip()

        # Skip empty chunks
        if not content:
            continue

        chunks.append({
            "article": article,
            "chapter": current_chapter,
            "content": content
        })


    return chunks

In [6]:
# function to parse pdf (2026 year) into chunks with metadata

def parse_f1_regulations_2026(full_text: str):


    # =====================================================
    # TEXT CLEANING
    # =====================================================

    # Fix common PDF ligatures
    full_text = (
        full_text
        .replace("ﬁ", "fi")
        .replace("ﬂ", "fl")
        .replace("\r", "\n")
    )

    # Patterns for headers / footers that should be removed
    skip_patterns = [
        r"SECTION B: SPORTING REGULATIONS",
        r"2026 Formula 1: Sporting Regulations",
        r"Fédération Internationale de l’Automobile",
        r"^Issue\s+\d+",
        r"^\d+\s+[A-Z]$",                 
        r"^\d{1,2}\s+\w+\s+\d{4}$",       
        r"^[A-Z]\d+$"                     
    ]

    lines = []

    for line in full_text.splitlines():

        line = line.strip()

        if not line:
            continue

        if any(re.search(pattern, line) for pattern in skip_patterns):
            continue

        lines.append(line)



    # =====================================================
    # PARSING STATE
    # =====================================================

    chunks = []

    current_chapter = None
    current_section = None
    pending_section = None

    current_article = None
    current_content = []



    # =====================================================
    # SAVE CURRENT ARTICLE
    # =====================================================

    def save_current_article():

        if current_article is None:
            return

        content = re.sub(
            r"\s+",
            " ",
            " ".join(current_content)
        ).strip()

        if not content:
            return

        chunks.append({
            "article": current_article,
            "chapter": f"{current_chapter}, {current_section}",
            "content": content
        })



    # =====================================================
    # MAIN PARSER LOOP
    # =====================================================

    i = 0

    while i < len(lines):

        line = lines[i]

        match = re.match(
            r"^ARTICLE\s+[A-Z]\d+:\s+(.+)$",
            line
        )

        if match:

            save_current_article()

            current_article = None
            current_content = []

            current_chapter = match.group(1).strip()

            i += 1
            continue

        match = re.match(
            r"^([A-Z]\d+\.\d+)\s+(.+)$",
            line
        )

        if match:

            pending_section = match.group(2).strip()

            i += 1
            continue

        match = re.match(
            r"^([A-Z]\d+\.\d+)$",
            line
        )

        if match and i + 1 < len(lines):

            next_line = lines[i + 1]

            if not re.match(
                r"^[A-Z]\d+\.\d+\.\d+$",
                next_line
            ):
                pending_section = next_line.strip()

                i += 2
                continue

        match = re.match(
            r"^([A-Z]\d+\.\d+\.\d+)$",
            line
        )

        if match:

            save_current_article()

            if pending_section:
                current_section = pending_section
                pending_section = None

            current_article = match.group(1)
            current_content = []

            i += 1
            continue

        if current_article:
            current_content.append(line)

        i += 1

    save_current_article() # save last article


    return chunks

Since the documents have a clear structure, it is possible to split them into chunks based on their structure. However, the PDF files from 2018–2025 have a different structure than the PDF file from 2026. For this reason, two different parsing functions were created.

In [7]:
# creating chunks

chunks = []

for pdf_name in metadata:

    first_page = metadata[pdf_name]['first_page']
    last_page = metadata[pdf_name]['last_page']
    full_text = extract_text(f'../data/documents/PDFs/{pdf_name}', first_page, last_page)

    year = metadata[pdf_name]['year'] 

    current_chunks = []

    if year >= 2018 and year <= 2025:
        current_chunks = parse_f1_regulations_2018_2025(full_text)
    elif year == 2026:
        current_chunks = parse_f1_regulations_2026(full_text)

    current_chunks = list( map( lambda x: {'year': year} | x , current_chunks ) )
    chunks += current_chunks


In [8]:
print(json.dumps(chunks, indent=4, ensure_ascii=False))

[
    {
        "year": 2018,
        "article": "1.1",
        "chapter": "REGULATIONS",
        "content": "The final text of these Sporting Regulations shall be the English version which will be used should any dispute arise as to their interpretation. Headings in this document are for ease of reference only and do not form part of these Sporting Regulations."
    },
    {
        "year": 2018,
        "article": "1.2",
        "chapter": "REGULATIONS",
        "content": "These Sporting Regulations apply to the Championship taking place in the calendar year referred to in the title (“the Championship”) and may only be changed after 30 April of the preceding year with the unanimous agreement of all competitors, save for changes made by the FIA for safety reasons which may come into effect without notice or delay."
    },
    {
        "year": 2018,
        "article": "2.1",
        "chapter": "GENERAL UNDERTAKING",
        "content": "All drivers, competitors and officials participa

Each chunk contains metadata about the regulation, including the year of the document, the chapter name, the article number, and the corresponding text content.

## 2. Creating vector database

As a vector database, we use Qdrant because it is fast, easy to use, and supports both dense and sparse embeddings, which are needed for hybrid search.

In [9]:
# creating vector database

qdrant_client = QdrantClient(path='../data/qdrant_storage')

In [10]:
# сreating collection

qdrant_client.create_collection(
    collection_name="regulations",
    vectors_config={
        "dense": VectorParams(
            size=384,
            distance=Distance.COSINE
        )
    },
    sparse_vectors_config={
        "sparse": SparseVectorParams()
    }
)

True

## 3. Creating embeddings

Since we use hybrid search, we need two types of vectors: dense and sparse. To generate them, we use the local models **BAAI/bge-small-en-v1.5** and **Qdrant/minicoil-v1**, respectively.

To improve retrieval quality, embeddings are created not only from the regulation text itself but also from its metadata, including the year, chapter name, and article number. This metadata is also stored in the payload together with each vector.

To make the system easier to extend in the future, an additional field indicating the regulation type is stored in the payload. Since the current dataset contains only Sporting Regulations, the value is set to **"sporting"**. This will allow Technical and Financial Regulations to be added later without changing the database structure.


In [11]:
dense_model = TextEmbedding(
    model_name="BAAI/bge-small-en-v1.5"
)

sparse_model = SparseTextEmbedding(
    model_name="Qdrant/minicoil-v1"
)


points = []

for chunk in chunks:

    year, article, chapter, content = chunk['year'], chunk['article'], chunk['chapter'], chunk['content']

    text = f"""
    Regulation Type: Sporting Regulation
    Year: {year}
    Article: {article}
    Chapter: {chapter}

    {content}
    """.strip()


    dense_embedding = next(dense_model.embed([text])).tolist()


    sparse_embedding = next(sparse_model.embed([text]))

    sparse_embedding_values = sparse_embedding.values.tolist()
    sparse_embedding_indices = sparse_embedding.indices.tolist()


    points.append(
                PointStruct(
                    id=str(uuid4()),
                    vector={
                        "dense": dense_embedding,
                        "sparse": {
                            "indices": sparse_embedding_indices,
                            "values": sparse_embedding_values
                        }
                    },
                    payload={'regulation_type': 'sporting'} | chunk
                )
    )


## 4. Saving embeddings into database

In [12]:
# load created data points into the database

qdrant_client.upsert(
    collection_name='regulations',
    points=points
)

UpdateResult(operation_id=0, status=<UpdateStatus.COMPLETED: 'completed'>)